# 🧠 AquaVolt-AI: LSTM Water Deficit Forecasting
This notebook trains a Long Short-Term Memory (LSTM) Recurrent Neural Network to forecast agricultural water deficits using live telemetry data generated by AquaVolt-AI.

In [ ]:
# Setup & Dependencies
!pip install -q pandas numpy matplotlib seaborn scikit-learn tensorflow

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as plt_sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Styling
plt.style.use('dark_background')


## 📡 Step 1: Load Live Dataset
We will pull the exact dataset generated by the `aquavolt_gsheet_logger.py` directly from Google Sheets.

In [ ]:
SHEET_ID = '1c2a-3t8fF2g_PX_0ape4ASTsbr5uX0Zb6YPzT8jtuN8'
csv_url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet=Sheet1'

print("Downloading live data...")
df = pd.read_csv(csv_url, low_memory=False)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)

# Select features
features = ['air_temp', 'solar_rad', 'humidity', 'ndvi']
target = 'water_need'

for c in features + [target]:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.dropna(subset=features + [target])
print(f"Loaded {len(df)} cleaned records for training.")


## ⚙️ Step 2: Sequence Preparation
LSTMs require sequential time-series windows (e.g., look at the last 24 hours to predict the next hour).

In [ ]:
LOOKBACK = 24 # 24 hours

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaled_X = scaler_X.fit_transform(df[features])
scaled_y = scaler_y.fit_transform(df[[target]])

X, y = [], []
for i in range(len(scaled_X) - LOOKBACK):
    X.append(scaled_X[i:i+LOOKBACK])
    y.append(scaled_y[i+LOOKBACK])

X, y = np.array(X), np.array(y)
print(f"Generated {len(X)} sequences of length {LOOKBACK}")

# Train/Test Split (80/20)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]


## 🧠 Step 3: Build & Train LSTM Model

In [ ]:
model = Sequential([
    LSTM(64, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print('\nTraining LSTM...')
history = model.fit(
    X_train, y_train,
    epochs=50, batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)


## 📊 Step 4: Evaluate Forecasts

In [ ]:
y_pred = model.predict(X_test)
y_pred_inv = scaler_y.inverse_transform(y_pred)
y_test_inv = scaler_y.inverse_transform(y_test)

r2 = r2_score(y_test_inv, y_pred_inv)
rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))

print(f"=== Forecast Performance ===")
print(f"R² Score: {r2:.3f}")
print(f"RMSE: {rmse:.3f} mm")

fig, ax = plt.subplots(figsize=(12, 5), facecolor='#0e1117')
ax.set_facecolor('#1a1a2e')
ax.plot(y_test_inv[:200], color='#4fc3f7', linewidth=1.5, label='Actual Water Deficit', alpha=0.8)
ax.plot(y_pred_inv[:200], color='#ef5350', linewidth=1.5, label='LSTM Predicted', alpha=0.8)
ax.set_title(f'LSTM Forecast vs Actual (R²={r2:.3f})', color='white', fontweight='bold')
ax.legend()
plt.show()
